In [0]:
%sql
WITH pocos AS (
    SELECT poco_id, nome_poco_anp
    FROM workspace.oil_gas_gold.dim_poco
    WHERE nome_poco_anp IN (
        '7-JUB-64D-ESS',
        '7-JUB-73D-ESS',
        '7-JUB-74D-ESS'
    )
),
meses AS (
    SELECT data AS competencia
    FROM workspace.oil_gas_gold.dim_data
    WHERE ano = 2025
      AND eh_primeiro_dia_mes = TRUE
)
SELECT
    m.competencia,
    p.nome_poco_anp,

    v.producao_de_oleo_m3 AS oleo_reportado_m3,
    v.oleo_m3_dia_calendario,

    CASE
        WHEN v.poco_id IS NULL
            THEN 'SEM_REGISTRO_NO_MES'
        ELSE v.status_informacao_oleo
    END AS status_informacao

FROM pocos p
CROSS JOIN meses m

LEFT JOIN workspace.oil_gas_gold.vw_producao_poco_mes_historico_v2 v
    ON v.escopo_id = 'historico_anp_v2'
    AND v.poco_id = p.poco_id
    AND v.competencia = m.competencia

ORDER BY m.competencia, p.nome_poco_anp

competencia,nome_poco_anp,oleo_reportado_m3,oleo_m3_dia_calendario,status_informacao
2025-01-01,7-JUB-64D-ESS,null,null,SEM_REGISTRO_NO_MES
2025-01-01,7-JUB-73D-ESS,null,null,SEM_REGISTRO_NO_MES
2025-01-01,7-JUB-74D-ESS,null,null,SEM_REGISTRO_NO_MES
2025-02-01,7-JUB-64D-ESS,null,null,SEM_REGISTRO_NO_MES
2025-02-01,7-JUB-73D-ESS,null,null,SEM_REGISTRO_NO_MES
2025-02-01,7-JUB-74D-ESS,null,null,SEM_REGISTRO_NO_MES
2025-03-01,7-JUB-64D-ESS,null,null,SEM_REGISTRO_NO_MES
2025-03-01,7-JUB-73D-ESS,null,null,SEM_REGISTRO_NO_MES
2025-03-01,7-JUB-74D-ESS,null,null,SEM_REGISTRO_NO_MES
2025-04-01,7-JUB-64D-ESS,17884.707000000,596.156900000,OLEO_INFORMADO_NOS_REGISTROS_DE_PRODUCAO


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
    competencia,
    producao_de_oleo_m3,
    oleo_m3_dia_calendario,
    pocos_com_id_distintos
FROM workspace.oil_gas_gold.vw_producao_campo_mes_historico_v2
WHERE escopo_id = 'historico_anp_v2'
  AND _source_dataset = 'producao_mar'
  AND ambiente = 'MAR'
  AND nome_campo_reportado = 'JUBARTE'
  AND bacia_reportada = 'CAMPOS'
  AND competencia BETWEEN DATE '2025-01-01' AND DATE '2025-12-01'
ORDER BY competencia

competencia,producao_de_oleo_m3,oleo_m3_dia_calendario,pocos_com_id_distintos
2025-01-01,609667.340000000,19666.688387097,51
2025-02-01,555407.397000000,19835.978464286,52
2025-03-01,586275.653000000,18912.117838710,51
2025-04-01,612086.249000000,20402.874966667,53
2025-05-01,678574.331000000,21889.494548387,52
2025-06-01,609795.686000000,20326.522866667,50
2025-07-01,815498.096000000,26306.390193548,52
2025-08-01,877092.704000000,28293.313032258,54
2025-09-01,919491.013000000,30649.700433333,54
2025-10-01,894159.814000000,28843.864967742,55


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
WITH volumes AS (
    SELECT
        f.poco_id,
        p.nome_poco_anp,

        SUM(CASE
            WHEN d.data = DATE '2025-11-01'
            THEN f.producao_de_oleo_m3
        END) AS oleo_novembro_m3,

        SUM(CASE
            WHEN d.data = DATE '2025-12-01'
            THEN f.producao_de_oleo_m3
        END) AS oleo_dezembro_m3

    FROM workspace.oil_gas_gold.vw_producao_historico_v2 f

    INNER JOIN workspace.oil_gas_gold.dim_data d
        ON f.data_id = d.data_id

    INNER JOIN workspace.oil_gas_gold.dim_poco p
        ON f.poco_id = p.poco_id

    WHERE f.escopo_id = 'historico_anp_v2'
      AND f._source_dataset = 'producao_mar'
      AND f.ambiente_reportado = 'MAR'
      AND f.campo_reportado = 'JUBARTE'
      AND f.bacia_reportada = 'CAMPOS'
      AND d.data IN (
          DATE '2025-11-01',
          DATE '2025-12-01'
      )

    GROUP BY f.poco_id, p.nome_poco_anp
),
taxas AS (
    SELECT
        *,
        CAST(
            oleo_novembro_m3 / 30 AS DECIMAL(28,9)
        ) AS taxa_novembro,
        CAST(
            oleo_dezembro_m3 / 31 AS DECIMAL(28,9)
        ) AS taxa_dezembro

    FROM volumes
    WHERE oleo_novembro_m3 IS NOT NULL
      AND oleo_dezembro_m3 IS NOT NULL
),
diferencas AS (
    SELECT
        *,
        taxa_dezembro - taxa_novembro
            AS diferenca_taxa_m3_dia_calendario
    FROM taxas
),
ordenados AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            ORDER BY diferenca_taxa_m3_dia_calendario, poco_id
        ) AS posicao
    FROM diferencas
)
SELECT
    CASE
        WHEN posicao <= 3 THEN nome_poco_anp
        ELSE 'DEMAIS POÇOS COM ÓLEO NOS DOIS MESES'
    END AS componente,

    COUNT(*) AS quantidade_pocos,

    SUM(diferenca_taxa_m3_dia_calendario)
        AS contribuicao_variacao_m3_dia,

    MIN(posicao) AS ordem

FROM ordenados

GROUP BY
    CASE
        WHEN posicao <= 3 THEN nome_poco_anp
        ELSE 'DEMAIS POÇOS COM ÓLEO NOS DOIS MESES'
    END

ORDER BY ordem

componente,quantidade_pocos,contribuicao_variacao_m3_dia,ordem
7-JUB-74D-ESS,1,-2457.603283871,1
7-JUB-64D-ESS,1,-2007.612806452,2
7-JUB-73D-ESS,1,-1590.154687097,3
DEMAIS POÇOS COM ÓLEO NOS DOIS MESES,35,1608.141101077,4


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
WITH contagens AS (
    SELECT
        ambiente,
        comportamento_observado,
        COUNT(*) AS intervencoes
    FROM workspace.oil_gas_gold.vw_comparacao_oleo_intervencoes_historico_v2
    WHERE escopo_id = 'historico_anp_v2'
      AND grupo_analise = 'RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO'
    GROUP BY ambiente, comportamento_observado
)
SELECT
    ambiente,

    CASE comportamento_observado
        WHEN 'DE_ZERO_PARA_POSITIVO'
            THEN 'De zero para positivo'
        WHEN 'DE_POSITIVO_PARA_ZERO'
            THEN 'De positivo para zero'
        WHEN 'TAXA_POSTERIOR_MAIOR'
            THEN 'Taxa posterior maior'
        WHEN 'TAXA_POSTERIOR_MENOR'
            THEN 'Taxa posterior menor'
        WHEN 'ZERO_NAS_DUAS_JANELAS'
            THEN 'Zero nas duas janelas'
        ELSE 'Taxas iguais na precisão adotada'
    END AS comportamento,

    intervencoes,

    SUM(intervencoes) OVER (
        PARTITION BY ambiente
    ) AS intervencoes_no_ambiente,

    ROUND(
        100.0 * intervencoes
        / SUM(intervencoes) OVER (PARTITION BY ambiente),
        2
    ) AS percentual_no_ambiente

FROM contagens
ORDER BY comportamento, ambiente

ambiente,comportamento,intervencoes,intervencoes_no_ambiente,percentual_no_ambiente
MAR,De positivo para zero,1,60,1.67
TERRA,De positivo para zero,12,1590,0.75
MAR,De zero para positivo,27,60,45.00
TERRA,De zero para positivo,96,1590,6.04
MAR,Taxa posterior maior,20,60,33.33
TERRA,Taxa posterior maior,820,1590,51.57
MAR,Taxa posterior menor,9,60,15.00
TERRA,Taxa posterior menor,618,1590,38.87
MAR,Zero nas duas janelas,3,60,5.00
TERRA,Zero nas duas janelas,44,1590,2.77


Databricks visualization. Run in Databricks to view.

In [0]:
displayHTML("""
<div style="font-family:Arial,sans-serif;padding:16px;
            color:#222;background:#fff;line-height:1.5">

  <h3 style="margin:0 0 10px">Como interpretar este painel</h3>

  <ul style="margin:0;padding-left:20px">
    <li>
      <b>Taxa de óleo:</b> volume reportado dividido pelos dias
      corridos do mês ou da janela; não representa vazão por dia operado.
    </li>
    <li>
      <b>Lacunas:</b> ausência de informação permanece sem valor.
      Zero é exibido somente quando reportado na fonte.
    </li>
    <li>
      <b>Escopos:</b> a curva de JUBARTE considera esse contexto de campo;
      as curvas dos três poços somam todos os seus contextos reportados.
      A decomposição de novembro para dezembro considera apenas JUBARTE.
    </li>
    <li>
      <b>Restaurações:</b> amostra do histórico ampliado, independente
      do caso JUBARTE. Inclui três meses elegíveis antes e três depois,
      excluindo os meses da intervenção, sem outro evento observado
      no intervalo completo. MAR: 60 eventos; TERRA: 1.590 eventos.
    </li>
    <li>
      <b>Interpretação:</b> mudanças observadas não comprovam causa
      operacional nem eficácia da intervenção.
    </li>
  </ul>

  <p style="margin:12px 0 0;color:#555;font-size:12px">
    Fonte: dados abertos da ANP · Escopo: historico_anp_v2
  </p>
</div>
""")

Como interpretar este painel 

 
 
 Taxa de óleo: volume reportado dividido pelos dias
 corridos do mês ou da janela; não representa vazão por dia operado.
 
 
 Lacunas: ausência de informação permanece sem valor.
 Zero é exibido somente quando reportado na fonte.
 
 
 Escopos: a curva de JUBARTE considera esse contexto de campo;
 as curvas dos três poços somam todos os seus contextos reportados.
 A decomposição de novembro para dezembro considera apenas JUBARTE.
 
 
 Restaurações: amostra do histórico ampliado, independente
 do caso JUBARTE. Inclui três meses elegíveis antes e três depois,
 excluindo os meses da intervenção, sem outro evento observado
 no intervalo completo. MAR: 60 eventos; TERRA: 1.590 eventos.
 
 
 Interpretação: mudanças observadas não comprovam causa
 operacional nem eficácia da intervenção.
 
 

 
 Fonte: dados abertos da ANP · Escopo: historico_anp_v2